# GIVP R - Benchmark Literature Comparison

Comparacao empirica entre **GIVP-full** e **GRASP-only** em 4 funcoes classicas, com 30 rodadas por configuracao.

## 1. Setup

In [8]:
suppressPackageStartupMessages({
  library(givp)
  library(dplyr)
})

# Niveis de execucao:
#   smoke  : n_runs=3,  n_dims=5,  max_iters=10  (< 1 min)
#   medium : n_runs=5,  n_dims=10, max_iters=30  (~5 min)
#   robust : n_runs=10, n_dims=10, max_iters=80  (~1h10min) <-- ATUAL
#   full   : n_runs=30, n_dims=10, max_iters=100 (~7 h)
n_runs    <- 10L  # robust
n_dims    <- 10L
max_iters <- 80L  # robust

cat(
  sprintf(
    "n_runs=%d, n_dims=%d, max_iters=%d\n",
    n_runs,
    n_dims,
    max_iters
  )
)

n_runs=10, n_dims=10, max_iters=80


## 2. Funcoes benchmark

In [9]:
sphere <- function(x) sum(x^2)

rosenbrock <- function(x) {
  sum(100 * (x[-1] - x[-length(x)]^2)^2 + (1 - x[-length(x)])^2)
}

rastrigin <- function(x) {
  10 * length(x) + sum(x^2 - 10 * cos(2 * pi * x))
}

ackley <- function(x) {
  n <- length(x)
  -20 * exp(-0.2 * sqrt(sum(x^2) / n)) -
    exp(sum(cos(2 * pi * x)) / n) + 20 + exp(1)
}

benchmarks <- list(
  Sphere = list(
    func = sphere,
    bounds = replicate(n_dims, c(-5.12, 5.12), simplify = FALSE),
    optimum = 0
  ),
  Rosenbrock = list(
    func = rosenbrock,
    bounds = replicate(n_dims, c(-5, 10), simplify = FALSE),
    optimum = 0
  ),
  Rastrigin = list(
    func = rastrigin,
    bounds = replicate(n_dims, c(-5.12, 5.12), simplify = FALSE),
    optimum = 0
  ),
  Ackley = list(
    func = ackley,
    bounds = replicate(n_dims, c(-32.768, 32.768), simplify = FALSE),
    optimum = 0
  )
)


## 3. Configuracoes dos algoritmos

In [10]:
cfg_full <- function(seed, max_iters = 80L) {
  givp::givp_config(
    max_iterations = max_iters,
    alpha = 0.12,
    adaptive_alpha = TRUE,
    alpha_min = 0.08,
    alpha_max = 0.18,
    vnd_iterations = 150L,  # smoke: 20L | medium: 60L | robust: 150L | full: 200L
    ils_iterations = 8L,    # smoke:  3L | medium:  5L | robust:   8L | full:  10L
    perturbation_strength = 4L,
    use_elite_pool = TRUE,
    elite_size = 7L,
    path_relink_frequency = 8L,
    use_cache = TRUE,
    cache_size = 10000L,
    early_stop_threshold = max_iters,
    use_convergence_monitor = TRUE,
    seed = seed
  )
}

cfg_grasp_only <- function(seed, max_iters = 80L) {
  givp::givp_config(
    max_iterations = max_iters,
    alpha = 0.12,
    adaptive_alpha = FALSE,
    vnd_iterations = 1L,
    ils_iterations = 1L,
    perturbation_strength = 0L,
    use_elite_pool = FALSE,
    use_convergence_monitor = FALSE,
    use_cache = TRUE,
    cache_size = 10000L,
    early_stop_threshold = max_iters,
    seed = seed
  )
}

## 4. Rodar experimento

In [11]:
records <- list()
idx <- 1L

total_tasks <- 2L * length(benchmarks) * n_runs
done <- 0L
experiment_t0 <- Sys.time()
checkpoint_path <- "R/benchmark_literature_comparison_r_partial.csv"

fmt_secs <- function(x) {
  if (!is.finite(x) || x < 0) return("--")
  h <- x %/% 3600
  m <- (x %% 3600) %/% 60
  s <- x %% 60
  sprintf("%02d:%02d:%05.2f", h, m, s)
}

cat(sprintf("Iniciando benchmark: %d execucoes totais\n", total_tasks))
flush.console()

for (algo in c("GIVP-full", "GRASP-only")) {
  for (fn_name in names(benchmarks)) {
    spec <- benchmarks[[fn_name]]
    cat(sprintf("\n>> Algoritmo=%s | Funcao=%s\n", algo, fn_name))
    flush.console()

    for (seed in 0:(n_runs - 1L)) {
      cfg <- if (algo == "GIVP-full") {
        cfg_full(seed, max_iters = max_iters)
      } else {
        cfg_grasp_only(seed, max_iters = max_iters)
      }

      t0 <- Sys.time()
      res <- givp::givp(
        spec$func,
        spec$bounds,
        config = cfg,
        direction = "minimize",
        seed = seed
      )
      elapsed <- as.numeric(difftime(Sys.time(), t0, units = "secs"))

      records[[idx]] <- data.frame(
        algorithm = algo,
        function_name = fn_name,
        seed = seed,
        fun = res$fun,
        nit = res$nit,
        nfev = res$nfev,
        time_s = elapsed
      )
      idx <- idx + 1L

      done <- done + 1L
      elapsed_total <- as.numeric(difftime(Sys.time(), experiment_t0, units = "secs"))
      avg_task <- elapsed_total / done
      eta <- avg_task * (total_tasks - done)

      cat(
        sprintf(
          "[%3d/%3d] %s | %s | seed=%d | fun=%.6g | run=%s | ETA=%s\n",
          done,
          total_tasks,
          algo,
          fn_name,
          seed,
          res$fun,
          fmt_secs(elapsed),
          fmt_secs(eta)
        )
      )
      flush.console()

      if (done %% 10L == 0L) {
        partial_df <- bind_rows(records)
        write.csv(partial_df, checkpoint_path, row.names = FALSE)
        cat(sprintf("Checkpoint salvo (%d linhas): %s\n", nrow(partial_df), checkpoint_path))
        flush.console()
      }
    }
  }
}

df <- bind_rows(records)
write.csv(df, checkpoint_path, row.names = FALSE)
cat(sprintf("Registros coletados: %d\n", nrow(df)))
cat(sprintf("Checkpoint final: %s\n", checkpoint_path))
head(df)

Iniciando benchmark: 80 execucoes totais

>> Algoritmo=GIVP-full | Funcao=Sphere
[  1/ 80] GIVP-full | Sphere | seed=0 | fun=0.012374 | run=00:02:38.51 | ETA=03:29:13.43
[  2/ 80] GIVP-full | Sphere | seed=1 | fun=0.0278168 | run=00:02:52.57 | ETA=03:35:28.23
[  3/ 80] GIVP-full | Sphere | seed=2 | fun=0.0233477 | run=00:03:30.64 | ETA=03:51:55.13
[  4/ 80] GIVP-full | Sphere | seed=3 | fun=0.00959665 | run=00:02:29.93 | ETA=03:39:09.49
[  5/ 80] GIVP-full | Sphere | seed=4 | fun=0.0230957 | run=00:03:20.06 | ETA=03:43:02.34
[  6/ 80] GIVP-full | Sphere | seed=5 | fun=0.0230291 | run=00:04:54.58 | ETA=04:03:56.69
[  7/ 80] GIVP-full | Sphere | seed=6 | fun=0.013742 | run=00:04:27.41 | ETA=04:12:45.25
[  8/ 80] GIVP-full | Sphere | seed=7 | fun=0.020852 | run=00:05:18.92 | ETA=04:25:58.50
[  9/ 80] GIVP-full | Sphere | seed=8 | fun=0.0214645 | run=00:04:24.23 | ETA=04:27:53.15
[ 10/ 80] GIVP-full | Sphere | seed=9 | fun=0.0309063 | run=00:03:50.39 | ETA=04:24:35.14
Checkpoint salvo (10 

,algorithm,function_name,seed,fun,nit,nfev,time_s
,<chr>,<chr>,<int>,<dbl>,<int>,<int>,<dbl>
1,GIVP-full,Sphere,0,0.012373976,80,56893,158.5097
2,GIVP-full,Sphere,1,0.027816784,80,56968,172.5698
3,GIVP-full,Sphere,2,0.023347731,80,55768,210.6369
4,GIVP-full,Sphere,3,0.009596649,80,56281,149.9250
5,GIVP-full,Sphere,4,0.023095744,80,56532,200.0632
6,GIVP-full,Sphere,5,0.023029126,80,56650,294.5844


## 5. Resumo

In [5]:
summary_df <- df |>
    group_by(function_name, algorithm) |>
    summarise(
        mean_fun = mean(fun),
        sd_fun = sd(fun),
        best_fun = min(fun),
        median_fun = median(fun),
        mean_nfev = mean(nfev),
        mean_time_s = mean(time_s),
        .groups = "drop"
    )

print(summary_df)


# A tibble: 8 × 8
  function_name algorithm    mean_fun     sd_fun   best_fun median_fun mean_nfev
  <chr>         <chr>           <dbl>      <dbl>      <dbl>      <dbl>     <dbl>
1 Ackley        GIVP-full      3.78       0.483      3.37       3.60       8183.
2 Ackley        GRASP-only    19.4        0.366     18.9       19.6         754 
3 Rastrigin     GIVP-full     48.9        6.77      42.0       48.8        6108 
4 Rastrigin     GRASP-only    85.2       14.0       73.9       75.6         758.
5 Rosenbrock    GIVP-full     30.8       12.2       19.1       24.0       10492.
6 Rosenbrock    GRASP-only 53734.     22511.     20202.     54448.          742.
7 Sphere        GIVP-full      0.0403     0.0178     0.0205     0.0392    10295.
8 Sphere        GRASP-only    26.9        7.13      18.1       24.3         754 
# ℹ 1 more variable: mean_time_s <dbl>


## 6. Wilcoxon

In [6]:
for (fn_name in unique(df$function_name)) {
  a <- df |>
    filter(function_name == fn_name, algorithm == "GIVP-full") |>
    arrange(seed) |>
    pull(fun)
  b <- df |>
    filter(function_name == fn_name, algorithm == "GRASP-only") |>
    arrange(seed) |>
    pull(fun)

  wt <- wilcox.test(a, b, paired = TRUE, alternative = "less", exact = FALSE)
  decision <- ifelse(wt$p.value < 0.05, "SIM", "NAO")
  cat(
    sprintf(
      "%s: W=%.1f p=%.4f %s\n",
      fn_name,
      wt$statistic,
      wt$p.value,
      decision
    )
  )
}


Sphere: W=0.0 p=0.0295 SIM
Rosenbrock: W=0.0 p=0.0295 SIM
Rastrigin: W=0.0 p=0.0295 SIM
Ackley: W=0.0 p=0.0295 SIM


## 7. Exportar JSON

In [7]:
jsonlite::write_json(
  list(
    metadata = list(n_runs = n_runs, n_dims = n_dims),
    summary = summary_df,
    records = df
  ),
  path = "R/benchmark_literature_comparison_r_results.json",
  pretty = TRUE,
  auto_unbox = TRUE
)
cat("Arquivo salvo: R/benchmark_literature_comparison_r_results.json\n")


Arquivo salvo: R/benchmark_literature_comparison_r_results.json
